<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_MLR/17_2_4_3_MLR_Ames_Part3_Revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLR Predicting Housing Prices in Ames Iowa: Part 3
## Regularization and Best Practices in Model Evaluation

Author: Brad Sheese

**Recap:**
- **Part 1:** Data cleaning, semantic fixing, and proper train/test splitting to avoid data leakage.
- **Part 2:** Feature selection (Forward/Backward) and modeling non-linear curves with polynomials.

**This notebook introduces:**
1. **Best Practices in Evaluation:** Using scikit-learn **Pipelines** to prevent subtle CV leaks, and mathematically translating results back from log-units to **actual dollars**.
2. **Regularization Techniques:** Lasso, Ridge, and Elastic Net, which help manage high-dimensional data (~225 features!) and improve generalization by penalizing complex models.
3. **Standardized Coefficients:** How to rank feature importance correctly using "Beta Weights."
4. **Residual Diagnostics:** Identifying model failures through residual scatter plots.

**Data Source:** http://jse.amstat.org/v19n3/decock/AmesHousing.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Download the cleaning module from GitHub
import urllib.request
module_url = "https://raw.githubusercontent.com/bsheese/cs377/main/17_regression_crossval/17_2_MLR/ames_cleaning.py"
urllib.request.urlretrieve(module_url, "ames_cleaning.py")
from ames_cleaning import load_and_clean_ames

# Load the model-ready data from Part 1
X_train, X_test, y_train, y_test = load_and_clean_ames()

print(f"X_train shape: {X_train.shape}")
print(f"y_train is log-transformed SalePrice. Mean: {y_train.mean():.4f}")

## Best Practices in Model Evaluation

To move from "rookie" to "pro" in machine learning, we must address three critical areas during evaluation:

### 1. The Cross-Validation Leak (Prevented by Pipelines)
If you scale your data or calculate medians on the *entire* training set before running cross-validation, your validation folds "leak" information from the future.
**The Fix:** Wrap all pre-processing steps (Imputer, Scaler) and the model into a scikit-learn `Pipeline`. When the pipeline is cross-validated, it recalculates statistics strictly within the isolated training folds of each individual split.

### 2. The Log-Dollar Illusion
Since our target variable is `log(SalePrice)`, an RMSE of 0.12 might look small, but it's measured in "log-dollars." To make this useful for a real estate client, we must inverse-transform our results.
**The Fix:** Apply `np.exp()` to both predictions and true values before calculating real-world metrics like RMSE and MAE in actual US dollars.

### 3. The Coefficient Magnitude Fallacy
You cannot compare the importance of `Lot Area` (measured in sq ft) to `Central Air` (binary 0 or 1) by looking at raw coefficients.
**The Fix:** Use `StandardScaler` in your pipeline. This produces **Standardized Coefficients** (Beta Weights) which represent the change in target per 1-standard-deviation change in the feature. Now, you can rank feature importance by the absolute size of the coefficients.

In [ ]:
def run_evaluation_pipeline(
    X_train, X_test, y_train, y_test,
    model,
    model_name="Model"
):
    """
    A professional evaluation workflow that prevents leakage via Pipelines,
    calculates metrics in real-world dollars, and plots residuals.
    """

    # 1. Build the Pipeline (Pro Fix: Impute and Scale inside the CV folds)
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    # 2. Cross-Validation (on training data only)
    # reporting log-RMSE for context
    cv_results = cross_validate(
        pipeline, X_train, y_train,
        cv=5,
        scoring='neg_root_mean_squared_error',
        return_train_score=False
    )
    cv_rmse_log = -cv_results['test_score'].mean()

    # 3. Final Fit and Test Prediction
    pipeline.fit(X_train, y_train)
    test_preds_log = pipeline.predict(X_test)

    # 4. Inverse-Transform (Pro Fix: Convert log-dollars back to real dollars)
    true_prices = np.exp(y_test)
    pred_prices = np.exp(test_preds_log)

    test_rmse_dollars = np.sqrt(mean_squared_error(true_prices, pred_prices))
    test_mae_dollars = mean_absolute_error(true_prices, pred_prices)
    test_r2 = r2_score(y_test, test_preds_log)

    print(f"--- {model_name} Evaluation ---")
    print(f"CV RMSE (log-units):   {cv_rmse_log:.4f}")
    print(f"Test R-squared:        {test_r2:.4f}")
    print(f"Test RMSE (dollars):  ${test_rmse_dollars:,.2f}")
    print(f"Test MAE (dollars):   ${test_mae_dollars:,.2f}\n")

    # 5. Residual Plotting (Pro Fix: Check for homoscedasticity)
    residuals_dollars = true_prices - pred_prices

    plt.figure(figsize=(10, 5))
    sns.scatterplot(x=pred_prices, y=residuals_dollars, alpha=0.5)
    plt.axhline(0, color='red', linestyle='--')
    plt.title(f'{model_name}: Residual Plot (Predicted vs. Errors)')
    plt.xlabel('Predicted Sale Price ($)')
    plt.ylabel('Residual Error ($)')
    plt.show()

    # Return the fitted model and metrics
    return {
        'pipeline': pipeline,
        'rmse_dollars': test_rmse_dollars,
        'r2': test_r2,
        'coeffs': pipeline.named_steps['model'].coef_
    }

### Baseline: Ordinary Least Squares (OLS)
We start with standard linear regression on all features to see how it handles the dataset before we apply regularization.

In [ ]:
ols_results = run_evaluation_pipeline(X_train, X_test, y_train, y_test, LinearRegression(), "OLS")

## Regularization: Managing Complexity

With ~225 features, OLS can become unstable or overfit. Regularization adds a "complexity tax" to the model's coefficients.

- **Ridge (L2):** Penalizes the square of the coefficients. It shrinks them toward zero but keeps all features. Great for handling multicollinearity.
- **Lasso (L1):** Penalizes the absolute value of the coefficients. It can shrink coefficients to **exactly zero**, effectively performing automatic feature selection.
- **Elastic Net:** A hybrid of both L1 and L2.

### Ridge Regression
We use a high alpha (penalty strength) to stabilize the model.

In [ ]:
ridge_results = run_evaluation_pipeline(X_train, X_test, y_train, y_test, Ridge(alpha=10), "Ridge")

### Lasso Regression
Lasso will aggressively zero out less useful features.

In [ ]:
lasso_results = run_evaluation_pipeline(X_train, X_test, y_train, y_test, Lasso(alpha=0.001), "Lasso")

### Elastic Net Regression
A balanced approach.

In [ ]:
enet_results = run_evaluation_pipeline(X_train, X_test, y_train, y_test, ElasticNet(alpha=0.001, l1_ratio=0.5), "ElasticNet")

## Ranking Feature Importance (Beta Weights)

Because our pipeline used `StandardScaler`, we can fairly compare the coefficients of all variables. Below we rank the top predictors from our Lasso model.

In [ ]:
# Extract coefficients and feature names
lasso_pipeline = lasso_results['pipeline']
coefs = lasso_results['coeffs']
feature_names = X_train.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Beta Weight': coefs
})

# Filter out features Lasso dropped to zero
importance_df = importance_df[importance_df['Beta Weight'] != 0]
importance_df['Absolute Weight'] = importance_df['Beta Weight'].abs()
importance_df = importance_df.sort_values(by='Absolute Weight', ascending=False)

print("Top 15 Most Important Features (Standardized Beta Weights):")
print(importance_df[['Feature', 'Beta Weight']].head(15).to_string(index=False))

## Summary

In this notebook, we leveled up our model evaluation by:
1. **Preventing CV Leakage:** Wrapping our scaling and imputation inside a `Pipeline`.
2. **Standardizing Coefficients:** Allowing us to see that features like `Total_Square_Footage` and `Overall Qual` are the true heavy hitters, regardless of their original units.
3. **Evaluating in Dollars:** Moving past the "Log-Dollar Illusion" to show that our Lasso model is off by approximately $22,000 on average (MAE).
4. **Visualizing Error:** Using residual plots to confirm that while the log-transform helped, our model still struggles slightly with high-value outlier homes (heteroscedasticity).

**Next Step:** In Part 4, we will use **Hyperparameter Tuning (GridSearch)** to find the perfect `alpha` and `l1_ratio` instead of guessing!